In [ ]:
# !git clone -b QCNN_integration https://github.com/merQlab/geqie.git

Cloning into 'geqie'...
remote: Enumerating objects: 1751, done.
remote: Counting objects: 100% (514/514), done.
remote: Compressing objects: 100% (191/191), done.
remote: Total 1751 (delta 386), reused 331 (delta 323), pack-reused 1237 (from 1)
Receiving objects: 100% (1751/1751), 53.10 MiB | 26.51 MiB/s, done.
Resolving deltas: 100% (968/968), done.


In [3]:
%cd ..

e:\Programowanko\Geqie\geqie


In [9]:
%cd geqie

[WinError 2] The system cannot find the file specified: 'geqie'
e:\Programowanko\Geqie\geqie\experiments


In [4]:
# !pip install -r requirements/requirements.txt --quiet

In [1]:
# !pip install pylatexenc

In [5]:
# !pip install .

In [ ]:
# !pip install qiskit_machine_learning --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 10.0 MB/s eta 0:00:00


In [4]:
from sklearn.datasets import make_classification
import matplotlib.pyplot as plt
import numpy as np
from torch import Tensor
from torch.nn import Linear, CrossEntropyLoss, MSELoss
from torch.optim import LBFGS
from matplotlib.colors import ListedColormap
from PIL import Image

In [5]:
# Load & process MNIST dataset:

from datasets import load_dataset
import os

current_dir = os.getcwd()
mnist_dataset = os.path.join("train-00000-of-00001.parquet")
# mnist_dataset = os.path.join(".MNIST_digits", "mnist", "train-00000-of-00001.parquet")
path_to_mnist_dataset = os.path.join(current_dir, mnist_dataset)

def load_and_process_mnist_dataset(labels_to_include=[0, 1], n_samples_per_label=100, resize=(8, 8)):
	mnist_dataset = load_dataset("parquet", data_files=path_to_mnist_dataset)

	selected_images = []
	for image_idx in range(len(mnist_dataset["train"])):
		# First, select only images with labels in labels_to_include:

		if mnist_dataset["train"][image_idx]["label"] in labels_to_include:
			img_8x8 = mnist_dataset["train"][image_idx]["image"].resize(resize, resample=Image.BILINEAR)
			img = np.array(img_8x8)
			selected_images.append({"image": img, "label": mnist_dataset["train"][image_idx]["label"]})

	X = np.array([item["image"] for item in selected_images])
	y = np.array([item["label"] for item in selected_images])

	return X, y

X, y = load_and_process_mnist_dataset(labels_to_include=[0, 1], n_samples_per_label=100, resize=(16, 16))

c:\Python\QML\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
import time

import glob
import torch
import numpy as np
import torch.nn as nn
from torch.nn import Linear, CrossEntropyLoss, MSELoss, NLLLoss
from torch.optim import LBFGS, Adam
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit import CircuitInstruction
from qiskit.circuit.library import UnitaryGate
from qiskit.primitives import StatevectorSampler as Sampler
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_machine_learning.neural_networks import SamplerQNN
from qiskit_machine_learning.gradients import ParamShiftSamplerGradient
from qiskit_machine_learning.connectors.torch_connector import _TorchNNFunction

import geqie
from geqie.encodings import frqi

class QNN_Pythorch_Module(nn.Module):
    def __init__(self, num_classes, num_qubits=8, num_layers=3):
        super().__init__()
        self.num_qubits = num_qubits
        self.num_classes = num_classes
        
        # 1. Create the parameterized VQC circuit ONCE
        self.vqc_circuit = QuantumCircuit(num_qubits)
        self.thetas = ParameterVector("theta", length=3 * num_qubits * num_layers)
        
        # Apply the brickwork architecture to the VQC circuit
        for layer in range(num_layers):
            offset = layer * 3 * num_qubits
            for i in range(num_qubits):
                self.vqc_circuit.rx(self.thetas[offset + i], i)
            for i in range(num_qubits):
                self.vqc_circuit.ry(self.thetas[offset + num_qubits + i], i)
            for i in range(num_qubits):
                self.vqc_circuit.rz(self.thetas[offset + 2*num_qubits + i], i)
            
            # Alternating brickwork entanglement
            if layer % 2 == 0:
                for i in range(0, num_qubits - 1, 2):
                    self.vqc_circuit.cx(i, i + 1)
            else:
                for i in range(1, num_qubits - 1, 2):
                    self.vqc_circuit.cx(i, i + 1)
                self.vqc_circuit.cx(num_qubits - 1, 0)
        
        # 2. Register the quantum weights directly as a native PyTorch Parameter.
        # Initialized from -pi to pi so the gates are fully active!
        self.quantum_weight = nn.Parameter(
            torch.empty(len(self.thetas)).uniform_(-np.pi, np.pi)
        )
        
        # 3. Classical head
        self.classical_head = nn.Linear(2**num_qubits, num_classes)

    def forward(self, matrix, default_shots=8192):
        # Build a clean, fresh circuit for this specific image safely
        qc = QuantumCircuit(self.num_qubits)
        qc.append(UnitaryGate(matrix), range(self.num_qubits))
        qc.compose(self.vqc_circuit, inplace=True)
        
        # Build the SamplerQNN using the active parameters
        fresh_sampler = Sampler(default_shots=default_shots)
        fresh_qnn = SamplerQNN(
            circuit=qc,
            input_params=None,
            weight_params=list(qc.parameters),
            sampler=fresh_sampler,
            gradient=ParamShiftSamplerGradient(sampler=fresh_sampler)
        )
        
        # Use raw PyTorch Autograd to bind our master weight to the QNN.
        # This completely bypasses TorchConnector's restrictive initialization.
        probs = _TorchNNFunction.apply(
            torch.empty(0),                 # input_data
            self.quantum_weight,  # weights (Our registered nn.Parameter!)
            fresh_qnn,            # neural_network
            False                 # sparse
        )
        # Feature scaling, drastically improves training time
        # Feels illegal, but it works, just don't let the gradient police know
        scaled_probs = probs * (2 ** self.num_qubits)
        
        # Classical classification
        logits = self.classical_head(scaled_probs)
        return torch.log_softmax(logits, dim=-1)

def train_QCNN(epochs=20, num_classes=2, circuits_directory="circuits"):
    f_loss = NLLLoss()
    qnn_model = QNN_Pythorch_Module(num_classes=num_classes, num_qubits=9, num_layers=1)
    
    optimizer = Adam([
        {'params': qnn_model.quantum_weight, 'lr': 0.001},
        {'params': qnn_model.classical_head.parameters(), 'lr': 0.01}
    ])
    
    qnn_model.train()
    loss_list = []
    batch_files = sorted(glob.glob(f"{circuits_directory}/batch_*.npz"))
    
    for epoch in range(epochs):
        start = time.time()
        total_loss = []
        
        for batch_file in batch_files:
            data = np.load(batch_file)
            matrices = data["matrices"]
            labels = data["labels"]
            
            optimizer.zero_grad() # Zero per batch
            
            for matrix, label in zip(matrices, labels):
                output = qnn_model(matrix)  
                label_tensor = torch.tensor(label, dtype=torch.long)
                # loss = f_loss(output, label_tensor)
                loss = f_loss(output, label_tensor) / len(matrices) 
                loss.backward()  

                total_loss.append(loss.item() * len(matrices))
                # total_loss.append(loss.item())

            optimizer.step() # Step once per batch

        loss_list.append(sum(total_loss) / len(total_loss))
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss_list[-1]:.4f}")
        end = time.time()
        print(f"Time: {end - start:.2f}s")

    return loss_list, qnn_model


In [ ]:
import numpy as np
import os
import geqie
from geqie.encodings import frqi
from qiskit.quantum_info import Operator
from qiskit import QuantumCircuit

def precompute_and_save_circuits(X_data, y_data, batch_size=5, save_dir="circuits"):
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
        
    num_samples = len(X_data)
    
    for i in range(0, num_samples, batch_size):
        batch_X = X_data[i : i + batch_size]
        batch_y = y_data[i : i + batch_size]
        
        batch_matrices = []
        
        for img in batch_X:
            # Encode the raw 0-255 uint8 image into a QuantumCircuit
            qc = geqie.encode(frqi.init_function, frqi.data_function, frqi.map_function, img)
            
            # Unpack the composite instructions into base gates
            flat_qc = qc.decompose()
            
            # Keep unpacking in case geqie nested them multiple layers deep
            while len(flat_qc.data) != len(flat_qc.decompose().data):
                flat_qc = flat_qc.decompose()
                
            # Filer out non-unitary gates
            pure_qc = QuantumCircuit(flat_qc.num_qubits)
            for instruction in flat_qc.data:
                # Handle different Qiskit version data structures safely
                op_name = instruction.operation.name if hasattr(instruction, 'operation') else instruction[0].name
                
                # Now that the 'reset' is exposed, this will catch and delete it!
                if op_name not in ['reset', 'measure', 'barrier']:
                    pure_qc.append(instruction)
            
            # Extract the exact unitary matrix using Qiskit Operator
            unitary_matrix = Operator(pure_qc).data
            
            # Cast as complex128 to preserve strict unitarity
            unitary_matrix = np.array(unitary_matrix, dtype=np.complex128)
            batch_matrices.append(unitary_matrix)
            
        # Save to .npz file
        batch_filename = os.path.join(save_dir, f"batch_{i//batch_size}.npz")
        np.savez(batch_filename, matrices=batch_matrices, labels=batch_y)
        
        print(f"Saved {batch_filename} with dtype {batch_matrices[0].dtype}")

In [13]:
precompute_and_save_circuits(X[:20], y[:20], batch_size=5)

Saved circuits_small\batch_0.npz with dtype complex64
Saved circuits_small\batch_1.npz with dtype complex64
Saved circuits_small\batch_2.npz with dtype complex64
Saved circuits_small\batch_3.npz with dtype complex64


In [6]:
mean1, mean2 = train_QCNN(epochs=10, num_classes=2)  # Train on a subset of the data for demonstration

Epoch 1/10, Loss: 2.3301
Time: 39.27s
Epoch 2/10, Loss: 0.3917
Time: 38.85s
Epoch 3/10, Loss: 0.3856
Time: 41.21s
Epoch 4/10, Loss: 0.0011
Time: 41.38s
Epoch 5/10, Loss: 0.0009
Time: 39.40s
Epoch 6/10, Loss: 0.0080
Time: 39.39s
Epoch 7/10, Loss: 0.0100
Time: 38.85s
Epoch 8/10, Loss: 0.0096
Time: 41.51s
Epoch 9/10, Loss: 0.0052
Time: 42.55s
Epoch 10/10, Loss: 0.0012
Time: 38.35s


In [7]:
mean11, mean22 = train_QCNN(epochs=25, num_classes=2)  # Train on a subset of the data for demonstration

Epoch 1/25, Loss: 2.1167
Time: 41.20s
Epoch 2/25, Loss: 0.3444
Time: 40.50s
Epoch 3/25, Loss: 0.0759
Time: 40.94s
Epoch 4/25, Loss: 0.0015
Time: 40.34s
Epoch 5/25, Loss: 0.0065
Time: 39.41s
Epoch 6/25, Loss: 0.0073
Time: 40.75s
Epoch 7/25, Loss: 0.0071
Time: 42.13s
Epoch 8/25, Loss: 0.0114
Time: 40.75s
Epoch 9/25, Loss: 0.0010
Time: 41.06s
Epoch 10/25, Loss: 0.0004
Time: 39.39s
Epoch 11/25, Loss: 0.0000
Time: 39.23s
Epoch 12/25, Loss: 0.0000
Time: 41.82s
Epoch 13/25, Loss: 0.0000
Time: 42.27s
Epoch 14/25, Loss: 0.0000
Time: 41.86s
Epoch 15/25, Loss: 0.0000
Time: 43.18s
Epoch 16/25, Loss: 0.0000
Time: 41.31s
Epoch 17/25, Loss: 0.0000
Time: 40.93s
Epoch 18/25, Loss: 0.0000
Time: 40.41s
Epoch 19/25, Loss: 0.0000
Time: 41.65s
Epoch 20/25, Loss: 0.0000
Time: 41.77s
Epoch 21/25, Loss: 0.0000
Time: 41.47s
Epoch 22/25, Loss: 0.0000
Time: 41.50s
Epoch 23/25, Loss: 0.0000
Time: 41.04s
Epoch 24/25, Loss: 0.0001
Time: 41.66s
Epoch 25/25, Loss: 0.0000
Time: 41.78s
